In [21]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

# Given data (theta, d, t, E):
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
])

# Top exploitation candidates (θ, d, t):
# [2.746 1.145 1.059]
# [2.245 1.043 1.008]
# [2.356 1.103 1.002]
# High-uncertainty candidate for exploration: [4.969 1.111 3.953]

X = data[:,0:3]   # design inputs
y = data[:,3]     # efficiency

# Fit GP with RBF kernel (normalized y):
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=[1,1,1], length_scale_bounds=(1e-2, 1e2))
gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gpr.fit(X, y)

# Generate candidate points (random sampling in bounds):
n_samples = 20000
theta_samples = np.random.uniform(0, 5, n_samples)
d_samples     = np.random.uniform(1, 3, n_samples)
t_samples     = np.random.uniform(1, 4, n_samples)
X_cand = np.vstack([theta_samples, d_samples, t_samples]).T

# GP predictions:
mean_pred, std_pred = gpr.predict(X_cand, return_std=True)

# Identify top candidates by predicted mean (exploitation):
top_indices = np.argsort(mean_pred)[-3:][::-1]  # sorted highest first
exploit_points = X_cand[top_indices]
exploit_mean = mean_pred[top_indices]
exploit_std  = std_pred[top_indices]

# Identify top candidate by uncertainty (exploration):
best_uncert_idx = np.argmax(std_pred)
explore_point = X_cand[best_uncert_idx]
explore_mean = mean_pred[best_uncert_idx]
explore_std  = std_pred[best_uncert_idx]

print("Top exploitation candidates (θ, d, t, mean, std):")
for pt, m, s in zip(exploit_points, exploit_mean, exploit_std):
    print(f"{np.round(pt,3)} -> mean={m:.2f}, std={s:.2f}")

print("\nHigh-uncertainty candidate for exploration:")
print(f"{np.round(explore_point,3)} -> mean={explore_mean:.2f}, std={explore_std:.2f}")


Top exploitation candidates (θ, d, t, mean, std):
[2.59  1.736 1.117] -> mean=95.44, std=1.09
[2.507 1.758 1.113] -> mean=95.09, std=0.90
[2.702 1.746 1.101] -> mean=95.00, std=0.84

High-uncertainty candidate for exploration:
[4.935 2.968 3.994] -> mean=42.74, std=26.66


In [2]:
import numpy as np
import plotly.express as px

# Given data (theta, d, t, E):
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
])

theta = data[:,0]
d     = data[:,1]
t     = data[:,2]
E     = data[:,3]

# 3D interactive scatter plot
fig = px.scatter_3d(
    x=theta, y=d, z=t,
    color=E,                       # color depends on efficiency
    color_continuous_scale="Viridis", # color map
    hover_data={"θ":theta, "d":d, "t":t, "E":E}, # hover info
    labels={"x":"θ (theta)", "y":"d (clearance)", "z":"t (tension)", "color":"Efficiency E"},
    title="3D Design Space with Efficiency"
)

# Make points larger for better visibility
fig.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey')))

# Show color bar on right
fig.update_layout(coloraxis_colorbar=dict(
    title="Efficiency E",
    thickness=20,
    tickvals=np.linspace(E.min(), E.max(), 6)
))

fig.show()


In [3]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Given data (theta, d, t, E):
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
])

theta = data[:,0]
d     = data[:,1]
t     = data[:,2]
E     = data[:,3]

# Final 3 chosen points
chosen_points = np.array([
    [2.746, 1.145, 1.059, "Exploit 1"],
    [2.356, 1.103, 1.002, "Exploit 2"],
    [4.969, 1.111, 3.953, "Explore"]
])

# Base plot with all original points
fig = px.scatter_3d(
    x=theta, y=d, z=t,
    color=E,
    color_continuous_scale="Viridis",
    hover_data={"θ":theta, "d":d, "t":t, "E":E},
    labels={"x":"θ (theta)", "y":"d (clearance)", "z":"t (tension)", "color":"Efficiency E"},
    title="3D Design Space with Efficiency and Selected New Points"
)

# Add the 3 chosen points as red markers
fig.add_trace(go.Scatter3d(
    x=chosen_points[:,0],
    y=chosen_points[:,1],
    z=chosen_points[:,2],
    mode="markers+text",
    marker=dict(size=8, color="red", symbol="diamond"),
    text=chosen_points[:,3],  # labels ("Exploit 1", etc.)
    textposition="top center",
    name="Chosen Points"
))

# Adjust colorbar
fig.update_layout(coloraxis_colorbar=dict(
    title="Efficiency E",
    thickness=20,
    tickvals=np.linspace(E.min(), E.max(), 6)
))

fig.show()


In [19]:
import numpy as np
import pandas as pd
import torch

# ---------------------------------------------------
# Simple custom neural network (manual autograd)
# ---------------------------------------------------
class NeuralNetwork:
    def __init__(self, input_size, hidden_layers, output_size):
        self.hidden_layers = hidden_layers
        self.input_size = input_size
        self.output_size = output_size

        # ---- First hidden layer (make weight tensors LEAF tensors) ----
        fan_in, fan_out = input_size, hidden_layers[0]
        xavier_std = (2.0 / (fan_in + fan_out)) ** 0.5
        W1 = torch.randn(fan_in, fan_out)
        W1.mul_(xavier_std)      # in-place scale -> stays a leaf
        W1.requires_grad_()      # now it's a leaf with requires_grad True
        self.W1 = W1

        b1 = torch.zeros(fan_out)
        b1.requires_grad_()
        self.b1 = b1

        # ---- Middle hidden layers ----
        for i in range(1, len(hidden_layers)):
            fan_in, fan_out = hidden_layers[i - 1], hidden_layers[i]
            xavier_std = (2.0 / (fan_in + fan_out)) ** 0.5
            w_temp = torch.randn(fan_in, fan_out)
            w_temp.mul_(xavier_std)
            w_temp.requires_grad_()
            b_temp = torch.zeros(fan_out)
            b_temp.requires_grad_()
            self.__setattr__(f"W{i + 1}", w_temp)
            self.__setattr__(f"b{i + 1}", b_temp)

        # ---- Output layer ----
        fan_in, fan_out = hidden_layers[-1], output_size
        xavier_std = (2.0 / (fan_in + fan_out)) ** 0.5
        w_temp = torch.randn(fan_in, fan_out)
        w_temp.mul_(xavier_std)
        w_temp.requires_grad_()
        b_temp = torch.zeros(fan_out)
        b_temp.requires_grad_()
        self.__setattr__(f"W{len(hidden_layers) + 1}", w_temp)
        self.__setattr__(f"b{len(hidden_layers) + 1}", b_temp)


    def forward(self, x):
        """Forward pass. Sigmoid activations for simplicity."""
        current_output = x
        for i in range(len(self.hidden_layers)):
            W = getattr(self, f"W{i + 1}")
            b = getattr(self, f"b{i + 1}")
            z = current_output @ W + b  # (batch x features) @ (features x next)
            current_output = torch.sigmoid(z)

        # Output layer
        W_out = getattr(self, f"W{len(self.hidden_layers) + 1}")
        b_out = getattr(self, f"b{len(self.hidden_layers) + 1}")
        z = current_output @ W_out + b_out
        output = torch.sigmoid(z)
        return output


    def backward(self, x, y, learning_rate):
        """
        Run forward, compute MSE loss, call backward(), then update
        leaf parameters in a safe way (inside torch.no_grad()).
        """
        output = self.forward(x)
        loss = torch.nn.functional.mse_loss(output, y)

        # compute gradients (populates .grad for leaf tensors)
        loss.backward()

        # Update parameters - do not perform arithmetic directly on a tensor
        # that requires grad without torch.no_grad(); instead use no_grad block.
        for i in range(len(self.hidden_layers) + 1):
            w = getattr(self, f"W{i + 1}")
            b = getattr(self, f"b{i + 1}")

            # Make sure gradients exist (they should for leaf tensors)
            if w.grad is None or b.grad is None:
                # This should not happen now, but guard just in case.
                raise RuntimeError(f"Missing gradient for W{i+1} or b{i+1}")

            with torch.no_grad():
                w -= learning_rate * w.grad
                b -= learning_rate * b.grad

            # Zero gradients safely
            w.grad.zero_()
            b.grad.zero_()

        return loss.item()


    def training_loop(self, x, y, iterations, learning_rate):
        for itr in range(iterations):
            loss = self.backward(x, y, learning_rate)
            if itr % 100 == 0:
                print(f"Iteration {itr}, Loss: {loss:.6f}")


# -------------------------------
# Main: prepare data and train
# -------------------------------
if __name__ == "__main__":
    # Your array (rows: features..., last column is target E)
    data = np.array([
        [3.19, 2.50, 2.44, 16.0],
        [2.11, 2.10, 2.93, 32.0],
        [2.86, 1.61, 3.18, 11.7],
        [3.07, 1.56, 2.71, 23.3],
        [2.95, 2.02, 2.77, 14.8],
        [2.20, 2.13, 3.04, 24.5],
        [2.95, 2.01, 3.16, 8.4 ],
        [2.62, 2.50, 3.75, 53.9],
        [0.07, 2.58, 3.72, 1.0 ],
        [4.95, 1.35, 1.35, 50.7],
        [0.48, 2.06, 1.32, 50.2],
        [4.78, 2.60, 1.83, 3.6 ],
        [2.54, 1.00, 1.00, 57.0],
        [0.57, 1.21, 3.68, 49.3],
        [4.95, 1.00, 1.00, 7.8 ],
        [2.50, 3.00, 3.30, 1.0 ],
        [0.00, 1.00, 2.20, 27.0],
        [2.75, 1.15, 1.06, 70.4],
        [2.36, 1.10, 1.00, 56.5],
        [4.97, 1.11, 3.95, 0.9],
        [1.95, 1.63, 1.00, 71.2],
        [1.39, 1.97, 2.27, 0.0],
        [5.0,  1.0,  4.0,  0.8],
        [2.70, 2.40, 1.16, 50.7],
        [2.51, 1.89, 1.08, 93.3],
        [1.79, 2.02, 4.00, 35.7],
        [2.49, 2.04, 1.00, 87.6],
        [2.74, 1.67, 1.00, 92.3],
        [2.18, 1.32, 1.33, 68.4],
        [2.76, 1.81, 1.03, 92.2],
        [2.56, 1.91, 1.09, 92.3],
        [2.93, 1.75, 1.11, 91.7],
    ])

    valve_designs = pd.DataFrame(data, columns=["theta", "d", "t", "E"])

    # Features and target (reshape y to (N,1) for MSE)
    X = valve_designs.drop(columns=["E"]).to_numpy()
    y = valve_designs["E"].to_numpy().reshape(-1, 1) / 100.0  # scale to 0..1

    # Convert to torch tensors
    x = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)

    # Initialize and train
    input_size = x.shape[1]         # number of features (3)
    hidden_layers = [16, 8]
    output_size = 1

    nn = NeuralNetwork(input_size, hidden_layers, output_size)

    iterations = 1000
    learning_rate = 0.01
    nn.training_loop(x, y, iterations, learning_rate)

    # Print actual vs predicted
    output = nn.forward(x)
    compare = torch.cat((y, output.detach()), dim=1)
    print("\nActual (scaled) vs Predicted")
    print(compare)


Iteration 0, Loss: 0.111483
Iteration 100, Loss: 0.108841
Iteration 200, Loss: 0.107487
Iteration 300, Loss: 0.106783
Iteration 400, Loss: 0.106390
Iteration 500, Loss: 0.106143
Iteration 600, Loss: 0.105962
Iteration 700, Loss: 0.105812
Iteration 800, Loss: 0.105676
Iteration 900, Loss: 0.105547

Actual (scaled) vs Predicted
tensor([[0.1600, 0.4204],
        [0.3200, 0.4154],
        [0.1170, 0.4205],
        [0.2330, 0.4218],
        [0.1480, 0.4203],
        [0.2450, 0.4158],
        [0.0840, 0.4203],
        [0.5390, 0.4172],
        [0.0100, 0.4041],
        [0.5070, 0.4284],
        [0.5020, 0.4035],
        [0.0360, 0.4257],
        [0.5700, 0.4193],
        [0.4930, 0.4068],
        [0.0780, 0.4286],
        [0.0100, 0.4155],
        [0.2700, 0.4024],
        [0.7040, 0.4203],
        [0.5650, 0.4181],
        [0.0090, 0.4281],
        [0.7120, 0.4145],
        [0.0000, 0.4109],
        [0.0080, 0.4282],
        [0.5070, 0.4173],
        [0.9330, 0.4175],
        [0.3570, 0.413

In [20]:
import numpy as np
import torch
import plotly.graph_objects as go
import plotly.express as px

# ------------------------------------
# 1. Prepare original data
# ------------------------------------
data = np.array([
    [3.19, 2.50, 2.44, 16.0],
    [2.11, 2.10, 2.93, 32.0],
    [2.86, 1.61, 3.18, 11.7],
    [3.07, 1.56, 2.71, 23.3],
    [2.95, 2.02, 2.77, 14.8],
    [2.20, 2.13, 3.04, 24.5],
    [2.95, 2.01, 3.16, 8.4 ],
    [2.62, 2.50, 3.75, 53.9],
    [0.07, 2.58, 3.72, 1.0 ],
    [4.95, 1.35, 1.35, 50.7],
    [0.48, 2.06, 1.32, 50.2],
    [4.78, 2.60, 1.83, 3.6 ],
    [2.54, 1.00, 1.00, 57.0],
    [0.57, 1.21, 3.68, 49.3],
    [4.95, 1.00, 1.00, 7.8 ],
    [2.50, 3.00, 3.30, 1.0 ],
    [0.00, 1.00, 2.20, 27.0],
    [2.75, 1.15, 1.06, 70.4],
    [2.36, 1.10, 1.00, 56.5],
    [4.97, 1.11, 3.95, 0.9],
    [1.95, 1.63, 1.00, 71.2],
    [1.39, 1.97, 2.27, 0.0],
    [5.0,  1.0,  4.0,  0.8],
    [2.70, 2.40, 1.16, 50.7],
    [2.51, 1.89, 1.08, 93.3],
    [1.79, 2.02, 4.00, 35.7],
    [2.49, 2.04, 1.00, 87.6],
    [2.74, 1.67, 1.00, 92.3],
    [2.18, 1.32, 1.33, 68.4],
    [2.76, 1.81, 1.03, 92.2],
    [2.56, 1.91, 1.09, 92.3],
    [2.93, 1.75, 1.11, 91.7],
])

theta = data[:, 0]
d = data[:, 1]
t = data[:, 2]
E = data[:, 3]

# ------------------------------------
# 2. Create a dense grid of new points
# ------------------------------------
theta_grid = np.linspace(theta.min(), theta.max(), 50)
d_grid = np.linspace(d.min(), d.max(), 50)
t_grid = np.linspace(t.min(), t.max(), 50)

grid = np.array(np.meshgrid(theta_grid, d_grid, t_grid)).T.reshape(-1, 3)
grid_tensor = torch.tensor(grid, dtype=torch.float32)

# ------------------------------------
# 3. Predict efficiency using trained NN
# ------------------------------------
with torch.no_grad():
    pred_scaled = nn.forward(grid_tensor)  # scaled (0–1)
    pred = pred_scaled * 100               # back to percentage

pred_np = pred.numpy().flatten()

# Find best predicted design
best_idx = np.argmax(pred_np)
best_point = grid[best_idx]
best_eff = pred_np[best_idx]

print(f"Best predicted design:")
print(f"θ = {best_point[0]:.3f}, d = {best_point[1]:.3f}, t = {best_point[2]:.3f},  Predicted E = {best_eff:.2f}%")

# ------------------------------------
# 4. Plot 3D scatter with best point
# ------------------------------------
fig = px.scatter_3d(
    x=theta, y=d, z=t, color=E,
    color_continuous_scale="Viridis",
    labels={"x":"θ (theta)", "y":"d (clearance)", "z":"t (tension)", "color":"Efficiency E"},
    title="NN-Predicted Design Space with Best Candidate"
)

# Add best predicted point
fig.add_trace(go.Scatter3d(
    x=[best_point[0]],
    y=[best_point[1]],
    z=[best_point[2]],
    mode="markers+text",
    marker=dict(size=8, color="red", symbol="diamond"),
    text=[f"Best NN: {best_eff:.1f}%"],
    textposition="top center",
    name="Best NN Prediction"
))

fig.update_layout(
    scene=dict(
        xaxis_title="θ (theta)",
        yaxis_title="d (clearance)",
        zaxis_title="t (tension)"
    ),
    coloraxis_colorbar=dict(title="Efficiency E"),
    legend=dict(x=0, y=1)
)

fig.show()


Best predicted design:
θ = 5.000, d = 1.000, t = 2.041,  Predicted E = 42.93%
